[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week07_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 7 Assignment: Multiple Testing & A/B Splitter Infrastructure

## QuickCart — Scaling the Experimentation Platform

QuickCart now runs **20+ simultaneous experiments** across their site. This creates two challenges:

1. **Multiple comparisons:** Each experiment tests multiple metrics. With 20 experiments and 10 metrics each, that's 200 tests — you'd expect ~10 false positives by chance alone. How do you separate real effects from noise?

2. **User assignment infrastructure:** Users must be deterministically assigned to experiments so that:
   - The same user always sees the same variant (no flickering)
   - Conflicting experiments never share users (e.g., two checkout page tests)
   - Assignment is ~50/50 balanced
   - Assignment is reproducible without storing state

In this assignment, you'll build both: a multiple testing correction pipeline and a hash-based experiment splitter.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import hashlib
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)

---

## Task 1: Multiple Testing Corrections

### Context

QuickCart ran an experiment and tested 20 metrics. The raw p-values are below. You know from the simulation setup that **only metrics 0, 7, and 15 have real effects** — the rest are null.

Your job: apply three correction methods and compare which effects each method discovers.

**Methods to implement:**

1. **Bonferroni:** Reject if $p_i < \alpha / m$ (controls Family-Wise Error Rate)
2. **Holm (step-down):** Sort p-values. Reject $p_{(i)}$ if $p_{(i)} < \alpha / (m - i + 1)$ for all $j \leq i$
3. **Benjamini-Hochberg (BH):** Sort p-values. Find largest $k$ where $p_{(k)} \leq k \cdot \alpha / m$. Reject all $p_{(1)}, ..., p_{(k)}$ (controls False Discovery Rate)

### Problem

1. Generate a simulated dataset: 20 metrics, where metrics 0, 7, and 15 have real effects (p-values near 0) and the rest are drawn from Uniform(0, 1)
2. Implement all three corrections
3. Report which effects each method discovers
4. Discuss the trade-offs: which method would you recommend for QuickCart's use case?

<details>
<summary>Hint 1: Generating realistic p-values</summary>

```python
n_metrics = 20
true_effects = [0, 7, 15]  # indices with real effects

# Null p-values are uniform
p_values = np.random.uniform(0, 1, n_metrics)

# True effects have small p-values
p_values[0] = 0.001   # strong effect
p_values[7] = 0.012   # moderate effect
p_values[15] = 0.035  # weak effect
```

</details>

<details>
<summary>Hint 2: Holm procedure step-by-step</summary>

```python
# Sort p-values (keep track of original indices)
sorted_indices = np.argsort(p_values)
sorted_pvals = p_values[sorted_indices]

# Step through sorted p-values
rejected = []
for i in range(m):
    threshold = alpha / (m - i)
    if sorted_pvals[i] <= threshold:
        rejected.append(sorted_indices[i])
    else:
        break  # stop at first non-rejection
```

</details>

<details>
<summary>Hint 3: Benjamini-Hochberg step-by-step</summary>

```python
# Sort p-values
sorted_indices = np.argsort(p_values)
sorted_pvals = p_values[sorted_indices]

# Find the largest k where p_(k) <= k * alpha / m
# Work backwards from m to 1
max_k = 0
for k in range(m, 0, -1):
    if sorted_pvals[k-1] <= k * alpha / m:
        max_k = k
        break

# Reject all p-values up to max_k
rejected = sorted_indices[:max_k].tolist()
```

</details>

In [ ]:
def bonferroni_correction(p_values, alpha=0.05):
    """Apply Bonferroni correction.
    
    Parameters
    ----------
    p_values : array-like
        Raw p-values from multiple tests.
    alpha : float
        Family-wise error rate to control.
    
    Returns
    -------
    list
        Indices of rejected hypotheses.
    """
    # YOUR CODE HERE
    pass


def holm_correction(p_values, alpha=0.05):
    """Apply Holm step-down correction.
    
    Parameters
    ----------
    p_values : array-like
        Raw p-values from multiple tests.
    alpha : float
        Family-wise error rate to control.
    
    Returns
    -------
    list
        Indices of rejected hypotheses.
    """
    # YOUR CODE HERE
    pass


def benjamini_hochberg_correction(p_values, alpha=0.05):
    """Apply Benjamini-Hochberg FDR correction.
    
    Parameters
    ----------
    p_values : array-like
        Raw p-values from multiple tests.
    alpha : float
        False discovery rate to control.
    
    Returns
    -------
    list
        Indices of rejected hypotheses.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test Task 1
np.random.seed(123)
n_metrics = 20
true_effects = [0, 7, 15]

# Generate p-values
p_values = np.random.uniform(0.1, 1.0, n_metrics)  # null p-values
p_values[0] = 0.001    # strong effect
p_values[7] = 0.012    # moderate effect
p_values[15] = 0.035   # weak effect (borderline)

print("Raw p-values:")
for i, p in enumerate(p_values):
    marker = " <-- TRUE EFFECT" if i in true_effects else ""
    print(f"  Metric {i:2d}: p = {p:.4f}{marker}")

# Apply corrections
rejected_bonf = bonferroni_correction(p_values)
rejected_holm = holm_correction(p_values)
rejected_bh = benjamini_hochberg_correction(p_values)

print(f"\nRejected by Bonferroni:          {sorted(rejected_bonf)}")
print(f"Rejected by Holm:                {sorted(rejected_holm)}")
print(f"Rejected by Benjamini-Hochberg:  {sorted(rejected_bh)}")
print(f"\nTrue effects at indices:          {true_effects}")

# Verify: Bonferroni should be most conservative
assert set(rejected_bonf).issubset(set(rejected_holm)), "Bonferroni should reject subset of Holm"
assert set(rejected_holm).issubset(set(rejected_bh)) or set(rejected_holm) == set(rejected_bh), \
    "Holm should generally reject subset of BH (or equal)"

# The strong effect (p=0.001) should be caught by all methods
assert 0 in rejected_bonf, "Bonferroni should detect the strong effect (p=0.001)"
assert 0 in rejected_holm, "Holm should detect the strong effect"
assert 0 in rejected_bh, "BH should detect the strong effect"

# No false positives should pass Bonferroni
false_positives_bonf = [i for i in rejected_bonf if i not in true_effects]
assert len(false_positives_bonf) == 0, f"Bonferroni has false positives: {false_positives_bonf}"

print("\nAll Task 1 checks passed!")

**Your recommendation:**

*Which correction method would you recommend for QuickCart? Consider: they care more about finding real improvements than about occasional false discoveries. Write 2-3 sentences.*

---

## Task 2: Implement the ABSplitter Class

### Context

QuickCart needs infrastructure that assigns each user to experiments **deterministically** using hashing. The system works in two stages:

1. **Slot assignment:** The experiment space is divided into `count_slots` slots. Each experiment occupies one or more slots. Users are hashed to a slot.
2. **Variant assignment:** Within each experiment, users are hashed to pilot or control (50/50).

Two different salts ensure the assignments are independent.

**Conflict handling:** Some experiments cannot share users (e.g., two experiments both modifying the checkout page). Conflicting experiments must be assigned to different slots.

### Problem

Implement the `ABSplitter` class with:

- `__init__`: Store slot count and salts
- `split_experiments`: Assign experiments to slots, respecting conflicts
- `process_user`: Determine which experiments a user is in and their variant

**Hash function:** `int(hashlib.md5(str.encode(value)).hexdigest(), 16) % modulo`

<details>
<summary>Hint 1: split_experiments algorithm</summary>

```python
# Strategy: assign experiments with most conflicts first (greedy)
# 1. Sort experiments by number of conflicts (descending)
# 2. For each experiment:
#    a. Find which slots are blocked (used by conflicting experiments)
#    b. Find available slots
#    c. Assign to the first available slot(s)
#    d. If no slots available, add to "unable to place" list
```

</details>

<details>
<summary>Hint 2: process_user logic</summary>

```python
def process_user(self, user_id):
    # Hash 1: user_id + salt_one -> slot number
    value_for_slot = str(user_id) + str(self.salt_one)
    slot = int(hashlib.md5(value_for_slot.encode()).hexdigest(), 16) % self.count_slots
    
    # Find which experiments are in this slot
    experiments_in_slot = [...]  # look up from split_experiments result
    
    # Hash 2: for each experiment, user_id + salt_two + experiment_id -> pilot/control
    results = []
    for exp_id in experiments_in_slot:
        value_for_variant = str(user_id) + str(self.salt_two) + str(exp_id)
        hash_val = int(hashlib.md5(value_for_variant.encode()).hexdigest(), 16) % 2
        variant = 'pilot' if hash_val == 0 else 'control'
        results.append((exp_id, variant))
    
    return slot, results
```

</details>

In [ ]:
class ABSplitter:
    """Hash-based A/B test splitter with conflict handling.
    
    Assigns users to experiment slots deterministically using MD5 hashing.
    Handles experiment conflicts by ensuring conflicting experiments
    never share the same slot.
    """
    
    def __init__(self, count_slots: int, salt_one: str, salt_two: str):
        """Initialize the splitter.
        
        Parameters
        ----------
        count_slots : int
            Total number of experiment slots available.
        salt_one : str
            Salt for slot assignment hash (user -> slot).
        salt_two : str
            Salt for variant assignment hash (user -> pilot/control).
        """
        # YOUR CODE HERE
        pass
    
    def split_experiments(self, experiments: list) -> list:
        """Allocate experiments to slots, respecting conflicts.
        
        Strategy: Assign experiments with the most conflicts first (greedy).
        Each experiment needs `count_slots` consecutive or available slots.
        
        Parameters
        ----------
        experiments : list of dict
            Each dict has keys:
            - 'experiment_id': unique identifier
            - 'count_slots': number of slots this experiment needs
            - 'conflict_experiments': list of experiment_ids that conflict
        
        Returns
        -------
        list
            List of experiment dicts that could NOT be placed (empty if all fit).
        """
        # YOUR CODE HERE
        pass
    
    def process_user(self, user_id) -> tuple:
        """Assign a user to a slot and determine their experiment variants.
        
        Parameters
        ----------
        user_id : str or int
            The user identifier.
        
        Returns
        -------
        tuple
            (slot_number, [(experiment_id, 'pilot'/'control'), ...])
        """
        # YOUR CODE HERE
        pass

In [ ]:
# Test Task 2: Basic functionality
splitter = ABSplitter(count_slots=10, salt_one='slot_salt', salt_two='variant_salt')

experiments = [
    {'experiment_id': 'exp_1', 'count_slots': 2, 'conflict_experiments': ['exp_2']},
    {'experiment_id': 'exp_2', 'count_slots': 2, 'conflict_experiments': ['exp_1']},
    {'experiment_id': 'exp_3', 'count_slots': 3, 'conflict_experiments': []},
    {'experiment_id': 'exp_4', 'count_slots': 1, 'conflict_experiments': []},
]

unable_to_place = splitter.split_experiments(experiments)
print(f"Unable to place: {unable_to_place}")
assert len(unable_to_place) == 0, "All experiments should fit in 10 slots"

# Test user processing
slot, assignments = splitter.process_user('user_12345')
print(f"\nUser 'user_12345':")
print(f"  Assigned to slot: {slot}")
print(f"  Experiment assignments: {assignments}")

assert 0 <= slot < 10, f"Slot should be in [0, 9], got {slot}"
for exp_id, variant in assignments:
    assert variant in ('pilot', 'control'), f"Variant must be 'pilot' or 'control', got {variant}"

# Determinism: same user should always get same assignment
slot2, assignments2 = splitter.process_user('user_12345')
assert slot == slot2 and assignments == assignments2, "Assignments must be deterministic!"

print("\nBasic Task 2 checks passed!")

---

## Task 3: Verify 50/50 Split Balance

### Context

A well-functioning splitter should assign approximately 50% of users to pilot and 50% to control for each experiment. Small deviations are expected (randomness), but systematic bias means the hash function isn't working properly.

### Problem

1. Process 10,000 synthetic user IDs through your splitter
2. For each experiment, count pilot vs. control assignments
3. Verify the split is approximately 50/50 (within 48-52%)
4. Also verify that slot assignment is approximately uniform (each slot gets ~1/N of users)

<details>
<summary>Hint</summary>

```python
n_users = 10000
pilot_counts = {}  # experiment_id -> count of pilot assignments
slot_counts = np.zeros(splitter.count_slots)

for i in range(n_users):
    user_id = f'user_{i}'
    slot, assignments = splitter.process_user(user_id)
    slot_counts[slot] += 1
    for exp_id, variant in assignments:
        if exp_id not in pilot_counts:
            pilot_counts[exp_id] = 0
        if variant == 'pilot':
            pilot_counts[exp_id] += 1
```

</details>

In [ ]:
# YOUR CODE HERE
#
# 1. Process 10,000 users
# 2. Track pilot/control counts per experiment
# 3. Track slot distribution
# 4. Print results and verify balance

n_test_users = 10000

# YOUR CODE HERE
pass

In [ ]:
# Verification (uncomment after completing above):

# # Check pilot/control balance for each experiment
# for exp_id, n_pilot in pilot_counts.items():
#     total_in_exp = total_counts[exp_id]  # total users in this experiment
#     if total_in_exp > 0:
#         pilot_frac = n_pilot / total_in_exp
#         print(f"{exp_id}: {pilot_frac:.1%} pilot ({n_pilot}/{total_in_exp})")
#         assert 0.47 < pilot_frac < 0.53, f"{exp_id} split is too imbalanced: {pilot_frac:.1%}"

# # Check slot uniformity
# expected_per_slot = n_test_users / splitter.count_slots
# for i, count in enumerate(slot_counts):
#     ratio = count / expected_per_slot
#     assert 0.85 < ratio < 1.15, f"Slot {i} has {count} users (expected ~{expected_per_slot:.0f})"

# print("\nTask 3 checks passed! Split is balanced.")

---

## Task 4: Verify Conflict Isolation

### Context

The most critical requirement: **conflicting experiments must never share users.** If experiment A and experiment B conflict, no user should be assigned to both.

### Problem

1. Set up experiments with known conflicts
2. Process 10,000 users and verify that no user appears in two conflicting experiments
3. Also test the edge case: what happens when conflicts make placement impossible?

<details>
<summary>Hint</summary>

```python
# Track which experiments each user is in
user_experiments = {}  # user_id -> set of experiment_ids

for i in range(10000):
    user_id = f'user_{i}'
    slot, assignments = splitter.process_user(user_id)
    exp_ids = {exp_id for exp_id, _ in assignments}
    user_experiments[user_id] = exp_ids

# Check no user is in both exp_1 and exp_2
for uid, exps in user_experiments.items():
    assert not ('exp_1' in exps and 'exp_2' in exps), \
        f"{uid} is in both conflicting experiments!"
```

</details>

In [ ]:
# YOUR CODE HERE
#
# 1. Use the splitter from Task 2 (exp_1 and exp_2 conflict)
# 2. Process 10,000 users
# 3. Verify no user appears in both conflicting experiments
# 4. Test edge case: too many experiments for available slots

# YOUR CODE HERE
pass

In [ ]:
# Edge case: impossible placement
# If we try to place 15 experiments of size 1 into 10 slots, 5 should fail

# splitter_small = ABSplitter(count_slots=5, salt_one='s1', salt_two='s2')
# many_experiments = [
#     {'experiment_id': f'exp_{i}', 'count_slots': 1, 'conflict_experiments': []}
#     for i in range(8)
# ]
# unable = splitter_small.split_experiments(many_experiments)
# print(f"Unable to place {len(unable)} experiments (expected 3)")
# assert len(unable) == 3, f"Expected 3 unplaced experiments, got {len(unable)}"
# print("Edge case check passed!")

---

## Summary

In this assignment you:
- Implemented three multiple testing correction methods (Bonferroni, Holm, BH)
- Understood the trade-off between controlling FWER vs. FDR
- Built a hash-based experiment assignment system with conflict handling
- Verified the system produces balanced, deterministic, conflict-free assignments

**Key takeaways:**
1. Always correct for multiple comparisons when testing many metrics. BH (FDR control) is usually the right choice for experimentation platforms — it's less conservative than Bonferroni while still controlling the false discovery rate.
2. Hash-based assignment is the industry standard for experiment infrastructure. It's deterministic (no database lookups), balanced, and handles conflicts naturally through slot isolation.